In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import torch
from torch.utils.data.dataloader import DataLoader
from data import PrepASV15Dataset, PrepASV19Dataset
import models
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
import sys

In [ ]:


def asv_cal_accuracies(protocol, path_data, net, device, data_type='time_frame', dataset=19,output_path=None):
    net = net.to(device)
    net.eval()
    with torch.no_grad():
        softmax_acc = 0
        num_files = 0
        probs = torch.empty(0, 3).to(device)

        if dataset == 15:
            test_set = PrepASV15Dataset(protocol, path_data, data_type=data_type)
        else:
            test_set = PrepASV19Dataset(protocol, path_data,args=None, data_type=data_type)

        test_loader = DataLoader(test_set, batch_size=24, shuffle=False, num_workers=4)
        # print(type(test_loader))
        zero_list=[]
        one_list=[]
        label_list=[]
        class_list=[]
        source_list=[]
        variant_list=[]
        start_time = time.time()
        for test_batch in tqdm(test_loader):
        # for test_batch in tqdm(test_loader, desc="Processing test batches"):
            # load batch and infer
            # print('abc')
            test_sample, test_label, sub_class,sub_source,sub_variant = test_batch

          
            
            
            num_files += len(test_label)

            test_sample = test_sample.to(device)
            test_label = test_label.to(device)
            # print(test_sample)
            infer = net(test_sample)
            # print(infer)
            # # print(type(infer))
            # sys.exit()
            
            t1 = F.softmax(infer, dim=1)
            t2 = test_label.unsqueeze(-1)
            row = torch.cat((t1, t2), dim=1)
      
            probs = torch.cat((probs, row), dim=0)
            zero_probs=probs[:,0].tolist()
            one_probs=probs[:,1].tolist()
            label_probs=test_label.tolist()
            class_probs=[i for i in sub_class]
            source_probs=[i for i in sub_source]
            variant_probs=[i for i in sub_variant]
            zero_list.extend(zero_probs)
            one_list.extend(one_probs)
            label_list.extend(label_probs)
            class_list.extend(class_probs)
            source_list.extend(source_probs)
            variant_list.extend(variant_probs)
            
            
            # for i in probs :
            #     print(i)
            # break
            
            infer = infer.argmax(dim=1)
            # print(infer)
            # print(type(infer))
            # for i in infer :
            #     print(i)
            # break
            batch_acc = infer.eq(test_label).sum().item()
            softmax_acc += batch_acc

        softmax_acc = softmax_acc / num_files
        end_time = time.time()
        print('Use time: {:.4f}s.'.format(end_time - start_time))
        with open(output_path,'w') as f:
            for z,o,l,c,s,v in zip(zero_list,one_list,label_list,class_list,source_list,variant_list):
                f.write(f'{z} {o} {l} {c} {s} {v}\n')
    return softmax_acc, probs.to('cpu')


def cal_roc_eer(probs, show_plot=True):
    """
    probs: tensor, number of samples * 3, containing softmax probabilities
    row wise: [genuine prob, fake prob, label]
    TP: True Fake
    FP: False Fake
    """
    all_labels = probs[:, 2]
  
    zero_index = torch.nonzero((all_labels == 0)).squeeze(-1)
    one_index = torch.nonzero(all_labels).squeeze(-1)
  
    zero_probs = probs[zero_index, 0]
    one_probs = probs[one_index, 0]
    # print(type(zero_probs))
    # for i in zero_probs:
    #     print(i)
    # for i in one_probs:
    #     print(i)

    threshold_index = torch.linspace(-0.1, 1.01, 10000)
    tpr = torch.zeros(len(threshold_index),)
    fpr = torch.zeros(len(threshold_index),)
    cnt = 0
    for i in threshold_index:

        tpr[cnt] = one_probs.le(i).sum().item()/len(one_probs)
        fpr[cnt] = zero_probs.le(i).sum().item()/len(zero_probs)
        cnt += 1

    sum_rate = tpr + fpr
    distance_to_one = torch.abs(sum_rate - 1)
    eer_index = distance_to_one.argmin(dim=0).item()
    out_eer = 0.5*(fpr[eer_index] + 1 - tpr[eer_index]).numpy()

    if show_plot:
        print('EER: {:.4f}%.'.format(out_eer * 100))
        plt.figure(1)
        plt.plot(torch.linspace(-0.2, 1.2, 1000), torch.histc(zero_probs, bins=1000, min=-0.2, max=1.2) / len(zero_probs))
        plt.plot(torch.linspace(-0.2, 1.2, 1000), torch.histc(one_probs, bins=1000, min=-0.2, max=1.2) / len(one_probs))
        plt.xlabel("Probability of 'Genuine'")
        plt.ylabel('Per Class Ratio')
        plt.legend(['Real', 'Fake'])
        plt.grid()

        plt.figure(3)
        plt.scatter(fpr, tpr)
        plt.xlabel('False Positive (Fake) Rate')
        plt.ylabel('True Positive (Fake) Rate')
        plt.grid()
        plt.show()

    return out_eer


if __name__ == '__main__':

    test_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    '''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
    '''
    data_path = 'change to your VoiceWukong dataset path'

    Net = models.SSDNet1D()
    
    model_path="change this to your Res-TSSDNet mdoel" # download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/ResTSSDNet.pth?download=true]
    check_point = torch.load(
        model_path
    )
   
        
    Net.load_state_dict(check_point['model_state_dict'])

    


/tmp/ipykernel_857725/1963938758.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  check_point = torch.load(


zh Use time: 249.7998s.

In [ ]:

protocol_file_path = 'change this to the zh_eval_list.txt'
accuracy, probabilities = asv_cal_accuracies(protocol_file_path, data_path, Net, test_device, data_type='time_frame', 
                                             dataset=19,
                                             output_path='change this to the path to save zh_eval_score.txt')
print(accuracy * 100)



print('End of Program.')
protocol_file_path = 'change this to the eval_list.txt'
accuracy, probabilities = asv_cal_accuracies(protocol_file_path, data_path, Net, test_device, data_type='time_frame', 
                                             dataset=19,
                                             output_path='change this to the path to save en_eval_score.txt')
print(accuracy * 100)



print('End of Program.')